In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
column_names = [
    "is-edible", 'cap-shape', "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring", "stalk-surface-below-ring",
    "stalk-color-above-ring", "stalk-color-below-ring", "veil-type", "veil-color",
    "ring-number", "ring-type", "spore-print-color", "population", "habitat"
]

df = pd.read_csv("C:\\Users\\casper\\OneDrive\\Masaüstü\\DS_project\\mushroom\\agaricus-lepiota.data",names= column_names) #dosya adresini değiştirmeyi unutmayın...

In [3]:
#FONKSİYONLAR

#Tüm rowlar için aynı değere sahip columları tespit eder. Bu columların olmadığı bi dataframe ve düzenlenmiş bir column_names listesi döndürür. Orijinalleri değiştirmez.

def dropIfNotUnique(df, dfNames):
    unique_counts = df.nunique()
    cols_to_drop = unique_counts[unique_counts == 1].index.tolist()
    
    if cols_to_drop:
        namesCopy = [col for col in dfNames if col not in cols_to_drop]
        print(f"Silinen sütunlar: {cols_to_drop}")
        return df.drop(columns=cols_to_drop), namesCopy
    else:
        namesCopy = column_names.copy()
        print("Veri setinde silinmeye uygun sütun bulunamadı.")
        return df, namesCopy
    
#Girilen adlara sahip columlara bakara ve aynı şekle sahip olmadıklarına bakar.
# Aralarında bi ilişki yoksa 0, varsa ve 0-0 1-1 şeklindeyse 1, varsa ve 1-0 0-1 şeklindeyse -1 döndürür
# İLK COLUMLARDA DEĞİL DUMMY/INDICATOR VARIABLEA DÖNÜŞTÜRÜLMÜŞ HALİNDE KULLANILMALIDIR.(0-1 lerden oluşan columlar)

def compareShape(col1,col2):
    uniquePairs = pd.DataFrame({"c1":col1 , "c2":col2}).drop_duplicates()

    if len(uniquePairs) == col1.nunique() and len(uniquePairs) == col2.nunique():
        if (col1[0] == col2[0]):
            return 1
        else:
            return -1
    else:
        return 0
    
#CompareShape fonksiyonunu tüm veriseti için çalıştırır ve ilişkili columnları bir dictionary haline getirip döndürür. 

def showRelations(dataFrame, dataColumnNames,dtlen): #keys will be the column names, and values will be -1 or 1.
                                               #-1 means they always have different values (one of them is 1 while other is 0) and 1 means they always have the same value.
    relationsDict = {}
    for i in range(dtlen):
        for j in range(i+1,dtlen):
            if(dataColumnNames[i][:-1] != dataColumnNames[j][:-1]):
                col1 = dataFrame[dataColumnNames[i]]
                col2 = dataFrame[dataColumnNames[j]]
                isSame = compareShape(col1,col2)
                if (isSame in [-1,1]):
                    columnsToAdd = (dataColumnNames[i] , dataColumnNames[j])
                    relationsDict[columnsToAdd] = isSame
    return relationsDict

In [ ]:
filtered, filteredColumnsName= dropIfNotUnique(df, column_names)

dummies = pd.get_dummies(filtered, columns = filteredColumnsName)
dummiesColumnsName = dummies.columns.values
end = len(dummies.columns)

relatedColumns = showRelations(dummies, dummiesColumnsName, end)
relatedColumnsToDrop = ["stalk-color-above-ring_c", "stalk-color-below-ring_c", "ring-number_n", "ring-type_n", "stalk-color-below-ring_o", "veil-color_y"]

dummiesCleaned = dummies.drop(columns = relatedColumnsToDrop) #İlişkili column kümelerinin her birinden 1er tane kalaak şekilde gerisi silindi.


In [ ]:
#Her columnun bar chartını görsel olarak kaydeder.
#Bu görselleri göndermiş olacağım. Bu kodu bir değişiklik yapmadığınız sürece kullanmanıza gerek yok.

for i in range(1,len(filtered)-1):
    plt.clf()
    pngName = filteredColumnsName[i] + "barChart.png"
    filtered[filteredColumnsName[i]].value_counts().plot(kind = 'bar', color="green")
    plt.savefig(pngName, dpi=300, bbox_inches="tight")